In [0]:
%run /Workspace/Users/antoniorad15@gmail.com/ROBOTICS-AI-training-pipeline/pipeline-finetune-gr00t/secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_4", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_4", 0o600)

ssh_user = dbutils.secrets.get(scope='brev', key='ssh_user').strip().splitlines()[-1]
os.environ['SSH_USER'] = ssh_user
print(f'SSH_USER: {ssh_user}')

In [ ]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
sudo mkdir -p /data/lerobot_datasets/merged_dataset
sudo chown -R $USER:$USER /data
EOF

In [ ]:
%sh
scp -r -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no \
  /Volumes/workspace/default/finetune_lerobot_datasets/$DATASET_NAME/* \
  $SSH_USER@$BREV_IP:/data/lerobot_datasets/merged_dataset/

In [0]:
import requests
import os
NEXT_JOB_ID = 449953930494301  

response = requests.post(
    f"{os.environ['DATABRICKS_HOST']}/api/2.1/jobs/run-now",
    headers={"Authorization": f"Bearer {os.environ['DATABRICKS_TOKEN']}"},
    json={"job_id": NEXT_JOB_ID}
)

if response.status_code == 200:
    print(f"Job 2 triggered: run_id={response.json()['run_id']}")
else:
    raise Exception(f"Failed to trigger Job 2: {response.text}")